## 1. LangGraph Orchestrator Graph Structure
This notebook explores the iterative harness controller implemented as a LangGraph StateGraph.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from dotenv import load_dotenv
load_dotenv()

from agents.orchestrator.agent import build_graph, orchestrator_graph
from agents.orchestrator.state import ResearchState
print("✅ Orchestrator graph loaded successfully")

## 2. Graph Topology
The orchestrator implements 7 nodes in a cycle: normalize → plan → execute → verify → observe → iterate/finalize.

In [ ]:
graph = build_graph()
print("📊 Graph nodes:")
for node in graph.get_graph().nodes:
    print(f"  - {node}")
print(f"\n🔗 Total nodes: {len(graph.get_graph().nodes)}")
print(f"📐 Graph structure: normalize → plan → execute → verify → observe → iterate|finalize")

## 3. ResearchState Schema
The state TypedDict carries all data across iterations — context accumulates, never discards.

In [ ]:
import inspect
from agents.orchestrator.state import ResearchState

print("📋 ResearchState fields:")
hints = ResearchState.__annotations__
for field, type_hint in hints.items():
    print(f"  {field}: {type_hint}")
print(f"\n📊 Total fields: {len(hints)}")

## 4. Run a Single Research Query
Execute the full harness loop on a sample query. Requires MCP servers running (backend auto-starts them).

In [ ]:
import uuid

result = await orchestrator_graph.ainvoke({
    "session_id": str(uuid.uuid4())[:12],
    "query": "What are the key techniques for semantic chunking of documents?",
    "file_path": "",
    "has_document": False,
    "iteration": 0,
    "max_iterations": 2,
    "quality_threshold": 6.0,
    "language_instruction": "You MUST respond entirely in English.",
    "research_plan": [],
    "accumulated_context": [],
    "current_draft": "",
    "verification_result": {},
    "verification_history": [],
    "quality_score": 0.0,
    "total_tokens": 0,
    "total_cost": 0.0,
    "failure_hints": "",
    "enable_web_search": False,
    "enable_planning": True,
    "enable_fact_check": True,
    "enable_parallel": True,
    "enable_sectioned": False,
    "report_sections": [],
    "section_order": [],
    "failing_sections": [],
    "status": "normalizing",
    "final_output": "",
    "error": "",
})

print(f"✅ Research complete!")
print(f"📊 Iterations: {result.get('iteration', 0)}")
print(f"⭐ Quality score: {result.get('quality_score', 0)}/10")
print(f"🔢 Total tokens: {result.get('total_tokens', 0):,}")
print(f"\n{'='*60}")
print(result.get("final_output", "No output")[:2000])

## 5. Iteration Decision Logic
The `should_iterate` function decides whether to continue or finalize.

In [ ]:
from agents.orchestrator.agent import should_iterate

scenarios = [
    {"quality_score": 8.0, "quality_threshold": 7.0, "iteration": 1, "max_iterations": 3},
    {"quality_score": 5.0, "quality_threshold": 7.0, "iteration": 1, "max_iterations": 3},
    {"quality_score": 5.0, "quality_threshold": 7.0, "iteration": 3, "max_iterations": 3},
]

print("🔄 Iteration decision scenarios:")
for i, s in enumerate(scenarios, 1):
    decision = should_iterate(s)
    print(f"  {i}. score={s['quality_score']}, threshold={s['quality_threshold']}, "
          f"iter={s['iteration']}/{s['max_iterations']} → {decision}")